<a href="https://colab.research.google.com/github/terry0809000/Artificial-Intelligence-KCL/blob/main/7PAVAIHA_2026_Practical_1_Draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 1: Downloading and utilising LLMS from Hugging Face

#IMPORTANT

Please save your own copy of this workbook and work off that. Click on `File` and select `Save a copy in Drive`

Then, click on `Runtime` above and if highlighted, click on `Restart session`

Finally, click on `edit` above and click on `Clear all outputs`

<hr style="border:1px solid black"> </hr>

For the first part of the practical, we want to make sure Google Colab is using the CPU runtime (not GPU).

Step 1: Open Runtime Settings
*   Click Runtime in the top menu.
*   Click Change runtime type.

Step 2: Select CPU, then click on Save

<hr style="border:1px solid black"> </hr>

#Set Up Hugging Face account and permissions

The aim of this pratical is to implement a local copy of a large language model (LLM)

The pratical is uses __[Hugging Face](https://huggingface.co/)__, an open source data science and machine learning platform to interface with LLM's

**Creating a Hugging Face Account and Access Token**

Before we can download a large language model (LLM), we need access to Hugging Face

First create a Hugging Face account (if you don't already have one) following the sign Up instuctions at https://huggingface.co/

Next, you need to generate an [Access Token](https://huggingface.co/docs/hub/en/security-tokens) so Colab can securely download models:


*   Click your profile picture (top right).
*   Click `Settings`. Then, in the left menu, click `Access Tokens`
*   Click `New token`

**Configure the Token**

Suggested settings when creating the token:

*   Name: 7PAVAIHA-practical
*   Role: Read

Click `Generate token`

*Important*: Copy the token immediately and store it somewhere safe. You will not be able to view it again.

**Log in from Colab**

Run the next two code block sto install the [`huggingface_hub`](https://github.com/huggingface/huggingface_hub) and log into you account

Paste your token when prompted.

If successful, you will see a confirmation message.

In [1]:
!pip install -q huggingface_hub

In [2]:
from huggingface_hub import login
login()  #hello

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


**Note** why [pip](https://www.w3schools.com/python/python_pip.asp) install in colab?

Installing dependencies explicitly ensures reproducibility, compatability, and consistency


Even if software libraries are already installed in colab, we install them again to guarantee that everyone is using the correct and most recent version


<hr style="border:1px solid black"> </hr>

## Download and run an LLM




We're going to utilise the [`transformers`](https://huggingface.co/docs/transformers/en/index). toolbox in Hugging Face as it provides a simple interface to download and utilise state-of-the-art pretrained AI models. You can read a blog on `transformers` [here](https://www.datacamp.com/tutorial/an-introduction-to-using-transformers-and-hugging-face).

In [3]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 93.5 MB/s eta 0:00:00


We import `[AutoTokenizer`](https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoTokenizer) and [`AutoModelForCausalLM`](https://huggingface.co/docs/transformers/en/model_doc/auto#transformers.AutoModelForCausalLM) which we can use to create classes of the models we are interested in using. You can read more about them is this [blog](https://community.ibm.com/community/user/blogs/ruslan-idelfonso-magaa-vsevolodovna/2023/10/05/how-to-work-with-pretrained-models-with-transforme).


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

We also need to import PyTorch (`torch`) as it's the engine that runs our models. It performs the tensor operations and matrix multiplications that generate text. You can read more on it [here](https://www.ibm.com/think/topics/pytorch).

In [5]:
import torch

Double check you are connected to CPU by running the next code block, if it does not state `False` please return to early instructions

In [6]:
print("CUDA available:", torch.cuda.is_available())

CUDA available: False


The next code block minimises the warnings you will get when implemententing the code

In [7]:
import warnings
warnings.filterwarnings("ignore")

We're now going to set up the [`Phi-3-mini-4k-instruct`](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) LLM.

Why this model?


*   It Is Open and Publicly Available
*   It is Small (But Still Powerful) so it's not too slow but has good reasoning ability
*   It's an "instruct" version, meaning the model has been fine-tuned to answer questions clearly

In [8]:
model_id = "microsoft/Phi-3-mini-4k-instruct"

First we need to set up a __[`tokenizer`](https://huggingface.co/docs/transformers/en/main_classes/tokenizer)__ which is required to prepare inputs (text) for inference in our model. You can read more on tokenization in this [blog](https://seantrott.substack.com/p/tokenization-in-large-language-models)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

This code downloads the pre-trained language model from Hugging Face and loads it into memory so it can be used.

**Question** How does it configure the model to run on the CPU with memory-efficient settings?

*   `device_map="cpu"` tells the model to load onto the CPU instead of the GPU.
*   `dtype=torch.float16` reduces numerical precision, meaning the model uses less memory per number.
*   `low_cpu_mem_usage=True` loads the model more efficiently by avoiding large temporary memory spikes during loading.

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print("Model loaded on CPU")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded on CPU


Now we set the question we'd like to ask the model (feel free to change this example) and the we generate the input tokens for this prompt

In [11]:
prompt = "What are possible uses of LLMS in healthcare: Keep in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

We can now feed these tokens into our model to generate the response

**Note:** This step is slow

In [12]:
output = model.generate(
    **inputs,
    max_new_tokens=128
)



We can then decode the response and print it

**Question:** Why does the answer get cut-off? What can you do to change this?

Language models generate text token by token. Note, A token is a small piece of text that a language model processes at one time. You can read more [here](https://huggingface.co/learn/llm-course/chapter2/4)

If the token limit is reached before the model produces an end-of-sequence token, the output will appear cut off.

Increase the limit - the code will get slower

In [13]:
response = tokenizer.decode(output[0], skip_special_tokens=False)
print(response)

What are possible uses of LLMS in healthcare: Keep in simple terms.

LLMS, or Learning Management Systems, are software applications designed to facilitate online learning and training. In healthcare, LLMS can be used for various purposes, including:

1. Medical education: LLMS can be used to deliver online courses, lectures, and training programs for medical students, residents, and practitioners. This can include topics such as anatomy, pharmacology, and clinical skills.

2. Continuing education: Healthcare professionals are required to participate in continuing education to maintain their licenses and certifications. LLMS can be used to


<hr style="border:1px solid black"> </hr>

## Having trouble get the code to run?

Not surprising, running LLM's is resouce heavy and when your colab allocats (low resource) CPU engines by default.

To overcome this, you can change colabs run time to a remote GPU

**Running Colabs GPU**

For the first part of the practical, we want to make sure Google Colab is using the CPU runtime (not GPU).

Step 1: Open Runtime Settings
*   Click Runtime in the top menu.
*   Click Change runtime type.

Step 2: Select `T4 GPU`, click 'ok' in the pop-up and then click on Save

This will reinitialize the session for with GPU computational resources.

**Question** Other than connecting to a GPU, what has happened to Colab after doing this?

*   The runtime has restarted. Colab has cleared all variables, deletes everything stored in memory, and resets the environment. You must re-run your notebook cells.


In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.1 MB/s eta 0:00:00


Re-import `torch` and check your run time type

Reconnect ot T$ if the porint out is not similer too

`CUDA available: True`

`GPU: Tesla T4`


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


Repeat earlier steps to set-up the tokenizer and model.

**Question** What has changed in the model code?


*   `device_map="auto"` tells Transformers to automatically place the model on the GPU if one is available.
*   `dtype=torch.float16` This sets the numerical precision to 16-bit floating point (FP16) instead of 32-bit. We're loading and running a higher resolution version ofthe model


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
model_id = "microsoft/Phi-3-mini-4k-instruct"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded


In [8]:
prompt = "What are possible uses of LLMS in healthcare: Keep in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

We can now feed these tokens into our model to generate the response

You can increase the value of `max_new_tokens` to generate more tokens as the model will be faster

In [9]:
output = model.generate(
    **inputs,
    max_new_tokens=512
)

response = tokenizer.decode(output[0], skip_special_tokens=False)
print(response)

What are possible uses of LLMS in healthcare: Keep in simple terms.

LLMS, or Learning Management Systems, are software applications designed to facilitate online learning and training. In healthcare, LLMS can be used for various purposes, including:

1. Medical education: LLMS can be used to deliver online courses, lectures, and training programs for medical students, residents, and practitioners. This can include topics such as anatomy, pharmacology, and clinical skills.

2. Continuing education: Healthcare professionals are required to participate in continuing education to maintain their licenses and certifications. LLMS can be used to deliver online courses and training programs that meet these requirements.

3. Patient education: LLMS can be used to create educational materials for patients, such as videos, articles, and interactive modules. These materials can help patients better understand their conditions, treatments, and medications.

4. Performance management: LLMS can be u

<hr style="border:1px solid black"> </hr>

## Set-up a LLM pipeline

We can use a Hugging Face [`pipeline`](https://huggingface.co/docs/transformers/en/main_classes/pipelines) to make this code more efficient. Piplelines are objects that abstract most of the complex code in hugging face, offering a simpler interface for several different taskk. you can read a blog on using `pipelines` [here](https://medium.com/@gdsc.srmramapuram/hugging-face-pipeline-a-simplified-overview-bbb37ba0ee1b).

In [10]:
from transformers import pipeline

In [11]:
generate_text = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id
)

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
res = generate_text("What are possible uses of Large Langauge Models in healthcare?")


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
print(res[0]["generated_text"])

What are possible uses of Large Langauge Models in healthcare? Langauge models are models that interpret and generate human language. They can be used for a variety of tasks, including:

1. Automated medical transcription: Language models can be used to transcribe medical recordings into written text, reducing the need for manual transcription and increasing efficiency and accuracy.

2. Medical chatbots: Language models can be used to create chatbots that provide medical advice and support to patients, helping to improve access to healthcare and reduce the burden on healthcare professionals.

3. Drug discovery: Language models can be used to analyze large amounts of scientific literature and identify potential drug targets, helping to accelerate the drug discovery process.

4. Medical research: Language models can be used to analyze and summarize scientific literature, helping researchers to identify trends and patterns in the data and develop new hypotheses.

5. Patient education: Lan

<hr style="border:1px solid black"> </hr>

## LLM Hyperparameters

**Question:** Find out what are the core three hyperparamter that effects how a LLLM chooses which tokens to samples when generating it's answers.

* `temperature`, `top_k`, `top_p`; you can read more about them [here](https://logicbig.com/tutorials/ai-tutorials/ai-foundations/llm-temperature.html#google_vignette)

**Time to code!**

*   Change the pipeline to implement one or more of these strategies
*   Systematically change their values and observed what happens

In [14]:
res = generate_text(
    "Why is AI so popular these days?",
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(res[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why is AI so popular these days?

The popularity of AI, or artificial intelligence, has surged in recent years due to a combination of technological advancements, economic factors, and societal needs. Here are some of the key reasons behind AI's growing popularity:

1. **Technological Advancements**: AI technology has made significant strides in recent years, becoming more powerful, accessible, and affordable. Developments in machine learning, deep learning, and natural language processing have enabled AI to perform complex tasks with increasing accuracy and efficiency.

2. **Data Availability**: The exponential growth of data, especially with the rise of the internet, social media, and IoT (Internet of Things), has provided AI systems with vast amounts of information to learn from. This data-rich environment has accelerated AI's development and application.

3. **Cost Reduction**: As AI technology becomes more mainstream, the cost of implementing AI solutions has decreased. This makes

<hr style="border:1px solid black"> </hr>

##  Using AI Assistants During the Exercises

In this module, you are permitted to use Gemini, ChatGPT, or Copilot to help you debug and write code. If you are doing this keep track of your prompts, and note down a short reflection of how helped you



Using the previous exercise as an example, as I already knew what the parameter were, I asked Copilot `"I want to adjust this pipeline to include temperature and top-p variable that will give more imaginative answers"` and supplied it a copy of the pipeline code.

It suggested the following code and gave me insights into what to expect when chaning these values.

In [15]:
res = generate_text("Why is AI so popular these days?",
                    do_sample=True,
                    temperature=1.1,
                    top_p=0.95)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Reflection**: Co-pilot gave me ready to run code, and the insights it provided helped me better set values for these parameters for more imaginative answers and helped me understand how those values had the desired effect.

<hr style="border:1px solid black"> </hr>

## Implement an altenative LLM

**Time to code!**

*   Alter the code to run an alternate LLM from Hugging Face. Ideally it should be an LLM suited for answering medical questions.

In [23]:
!pip -q install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

def ask_llm(question: str) -> str:
    prompt = (
        "You are a medical assistant. Provide clear, cautious medical information. "
        "If uncertain, say so.\n\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=1,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print(ask_llm("What are common causes of microcytic anemia?"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.0 MB/s eta 0:00:00


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


spleen cancer, spleen adenopathy, and spleen smears.


<hr style="border:1px solid black"> </hr>

*   Implement a pre-trained language translation model from Hugging Face


In [1]:
!pip -q install -U transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "facebook/nllb-200-distilled-600M"  # strong multilingual MT model

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

def translate(text: str, src_lang: str, tgt_lang: str, max_new_tokens: int = 256) -> str:
    """
    src_lang / tgt_lang are NLLB language codes, e.g.:
      English: "eng_Latn"
      Chinese (Simplified): "zho_Hans"
      Chinese (Traditional): "zho_Hant"
      Spanish: "spa_Latn"
      French: "fra_Latn"
    """
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

print(translate("How are you feeling today?", "eng_Latn", "zho_Hans"))
print(translate("患者是否应该有权拒绝AI参与？", "zho_Hans", "eng_Latn"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


今天你怎么感觉?
Should a patient have the right to refuse AI participation?


**Optional**

*   Connect Colab to your laptop's local runtime (as oposed to colab provided CPU or GPU)

<hr style="border:1px solid black"> </hr>